In [1]:
# NORTHSTAR — Solace Browser: Webservice Endpoints QA
# DNA: endpoint_qa = routes(health+schedule+evidence+budget+apps) x cors(no_wildcard) x auth(rate_limit+hmac) x structure(handler_class)
# File: web/server.py (Python HTTP server — file-based analysis, no running server)
# Towers: T1(Masters) T6(Hackers) T17(Performance) T23(Web)
# Auth: 65537 | File-based probes — reads Python source directly
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "solace-browser-webservice-endpoints-qa"
NOTEBOOK_PATH = "notebooks/qa/31-webservice-endpoints-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"
SERVER_PY = WEB / "server.py"

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

assert SERVER_PY.exists(), f"MISSING: {SERVER_PY}"
server_src = SERVER_PY.read_text(encoding="utf-8")

print(f"BOOTSTRAP: server.py loaded ({len(server_src)} chars, {len(server_src.splitlines())} lines)")
print(f"File: {SERVER_PY}")

BOOTSTRAP: server.py loaded (149426 chars, 3300 lines)
File: /home/phuc/projects/solace-browser/web/server.py


In [2]:
# P1: Server file exists, has HTTP handler class with do_GET/do_POST
print("=== P1: Server Structure ===")

# Has a request handler class (inherits from BaseHTTPRequestHandler or similar)
has_handler_class = bool(re.search(r'class\s+\w+.*(?:HTTPRequestHandler|BaseHTTPRequestHandler|SimpleHTTPRequestHandler)', server_src))
record("P1-handler-class", "server.py defines an HTTP request handler class",
       has_handler_class,
       detail="No class inheriting from HTTPRequestHandler found" if not has_handler_class else "",
       tower_ref="T1,T23")

# Has do_GET method
has_do_get = bool(re.search(r'def\s+do_GET\s*\(', server_src))
record("P1-do-get", "server.py has do_GET method", has_do_get,
       detail="No do_GET method found" if not has_do_get else "",
       tower_ref="T1,T23")

# Has do_POST method
has_do_post = bool(re.search(r'def\s+do_POST\s*\(', server_src))
record("P1-do-post", "server.py has do_POST method", has_do_post,
       detail="No do_POST method found" if not has_do_post else "",
       tower_ref="T1,T23")

# Uses ThreadingHTTPServer or similar (concurrent handling)
has_threading = bool(re.search(r'ThreadingHTTPServer|ThreadingMixIn|threading', server_src))
record("P1-threading", "server.py uses threaded server", has_threading,
       detail="No threading support found" if not has_threading else "",
       tower_ref="T1,T17")

# Has a main block or serve function
has_main = bool(re.search(r'if\s+__name__.*==.*__main__|def\s+(main|serve|run_server)', server_src))
record("P1-main", "server.py has main/serve entry point", has_main,
       detail="No __main__ block or serve function" if not has_main else "",
       tower_ref="T1,T23")

passed = sum(1 for r in results if r["id"].startswith("P1") and r["status"] == "PASS")
total_p1 = sum(1 for r in results if r["id"].startswith("P1"))
print(f"\nP1 complete: {passed}/{total_p1} passed")

=== P1: Server Structure ===
  PASS: server.py defines an HTTP request handler class
  PASS: server.py has do_GET method
  PASS: server.py has do_POST method
  PASS: server.py uses threaded server
  PASS: server.py has main/serve entry point

P1 complete: 5/5 passed


In [3]:
# P2: API routes exist for: /api/health, /api/schedule, /api/evidence, /api/budget, /api/apps
print("=== P2: Required API Routes ===")

REQUIRED_ROUTES = {
    "health": r'/api/(?:health|openapi\.json)',
    "schedule": r'/api/schedule',
    "evidence": r'/api/evidence',
    "budget": r'/api/budget',
    "apps": r'/api/apps',
}

for route_name, pattern in REQUIRED_ROUTES.items():
    found = bool(re.search(pattern, server_src))
    record(f"P2-route-{route_name}", f"server.py handles {route_name} route ({pattern})",
           found,
           detail=f"No route matching {pattern} found" if not found else "",
           tower_ref="T1,T23")

# Count total unique API routes
all_routes = set(re.findall(r'(?:request_path\s*==\s*|request_path\.startswith\()"(/api/[^"]+)"', server_src))
record("P2-route-count", f"server.py defines {len(all_routes)} unique API routes",
       len(all_routes) >= 10,
       detail=f"Only {len(all_routes)} routes, expected >=10" if len(all_routes) < 10 else "",
       tower_ref="T1,T23")

# Additional important routes
BONUS_ROUTES = {
    "cli-agents": r'/api/cli-agents',
    "yinyang-chat": r'/api/yinyang/chat',
    "settings": r'/api/settings',
    "cloud-sync": r'/api/cloud/sync',
}
for route_name, pattern in BONUS_ROUTES.items():
    found = bool(re.search(pattern, server_src))
    record(f"P2-bonus-{route_name}", f"server.py handles {route_name}",
           found,
           detail=f"Route {pattern} not found" if not found else "",
           tower_ref="T1,T23")

passed = sum(1 for r in results if r["id"].startswith("P2") and r["status"] == "PASS")
total_p2 = sum(1 for r in results if r["id"].startswith("P2"))
print(f"\nP2 complete: {passed}/{total_p2} passed")

=== P2: Required API Routes ===
  PASS: server.py handles health route (/api/(?:health|openapi\.json))
  PASS: server.py handles schedule route (/api/schedule)
  PASS: server.py handles evidence route (/api/evidence)
  PASS: server.py handles budget route (/api/budget)
  PASS: server.py handles apps route (/api/apps)
  PASS: server.py defines 44 unique API routes
  PASS: server.py handles cli-agents
  PASS: server.py handles yinyang-chat
  PASS: server.py handles settings
  PASS: server.py handles cloud-sync

P2 complete: 10/10 passed


In [4]:
# P3: CORS handling present (not wildcard)
print("=== P3: CORS Handling ===")

# CORS headers are set somewhere
has_cors = bool(re.search(r'Access-Control-Allow-Origin|CORS|cors', server_src, re.IGNORECASE))
record("P3-cors-present", "server.py has CORS handling", has_cors,
       detail="No CORS headers or handling found" if not has_cors else "",
       tower_ref="T6")

# No wildcard CORS (Access-Control-Allow-Origin: *)
wildcard_cors = re.findall(r"Access-Control-Allow-Origin[\"']?\s*[:,]\s*[\"']\*[\"']", server_src)
record("P3-no-wildcard", "server.py does NOT use wildcard CORS (*)",
       len(wildcard_cors) == 0,
       detail=f"Found {len(wildcard_cors)} wildcard CORS" if wildcard_cors else "",
       tower_ref="T6")

# Has CORS preflight handler (OPTIONS method or do_OPTIONS)
has_preflight = bool(re.search(r'do_OPTIONS|OPTIONS|preflight|CORS preflight', server_src))
record("P3-preflight", "server.py handles CORS preflight (OPTIONS)", has_preflight,
       detail="No OPTIONS/preflight handler found" if not has_preflight else "",
       tower_ref="T6")

# Origin is checked against allowlist (not just any origin)
has_origin_check = bool(re.search(r'localhost|127\.0\.0\.1|allowed.*origin|ALLOWED_ORIGIN|origin.*allow', server_src, re.IGNORECASE))
record("P3-origin-allowlist", "server.py restricts origins to localhost/allowlist",
       has_origin_check,
       detail="No origin restriction found" if not has_origin_check else "",
       tower_ref="T6")

passed = sum(1 for r in results if r["id"].startswith("P3") and r["status"] == "PASS")
total_p3 = sum(1 for r in results if r["id"].startswith("P3"))
print(f"\nP3 complete: {passed}/{total_p3} passed")

=== P3: CORS Handling ===
  PASS: server.py has CORS handling
  PASS: server.py does NOT use wildcard CORS (*)
  PASS: server.py handles CORS preflight (OPTIONS)
  PASS: server.py restricts origins to localhost/allowlist

P3 complete: 4/4 passed


In [5]:
# P4: Rate limiting or auth middleware present
print("=== P4: Rate Limiting + Auth ===")

# Rate limiting
has_rate_limit = bool(re.search(r'rate.?limit|_rate_limit|throttl', server_src, re.IGNORECASE))
record("P4-rate-limit", "server.py has rate limiting", has_rate_limit,
       detail="No rate limiting found" if not has_rate_limit else "",
       tower_ref="T6,T17")

# Rate limit uses timestamps / sliding window (not just a counter)
has_timestamp_rate = bool(re.search(r'time\.time|timestamps|sliding|per\s*minute', server_src))
record("P4-rate-limit-temporal", "Rate limiter uses time-based window",
       has_timestamp_rate,
       detail="Rate limiter may not use time-based window" if not has_timestamp_rate else "",
       tower_ref="T6")

# Auth: HMAC or token-based authentication
has_hmac = bool(re.search(r'hmac\.compare_digest|hmac|token.*auth|Authorization', server_src))
record("P4-auth-hmac", "server.py uses HMAC or token auth", has_hmac,
       detail="No HMAC or token authentication found" if not has_hmac else "",
       tower_ref="T6")

# CSRF protection (checks header or token)
has_csrf = bool(re.search(r'csrf|X-Requested-With|Origin.*check|referer.*check', server_src, re.IGNORECASE))
record("P4-csrf", "server.py has CSRF protection", has_csrf,
       detail="No CSRF protection found" if not has_csrf else "",
       tower_ref="T6")

# No broad except Exception: pass (FALLBACK BAN)
broad_catches = re.findall(r'except\s+Exception\s*:', server_src)
record("P4-no-broad-except", f"server.py has no broad except Exception ({len(broad_catches)} found)",
       len(broad_catches) == 0,
       detail=f"Found {len(broad_catches)} broad exception catches (FALLBACK BAN violation)" if broad_catches else "",
       tower_ref="T1,T6")

passed = sum(1 for r in results if r["id"].startswith("P4") and r["status"] == "PASS")
total_p4 = sum(1 for r in results if r["id"].startswith("P4"))
print(f"\nP4 complete: {passed}/{total_p4} passed")

=== P4: Rate Limiting + Auth ===
  PASS: server.py has rate limiting
  PASS: Rate limiter uses time-based window
  PASS: server.py uses HMAC or token auth
  PASS: server.py has CSRF protection
  PASS: server.py has no broad except Exception (0 found)

P4 complete: 5/5 passed


In [6]:
# P5: Evidence summary — aggregate results, compute score, seal hash
print("=== P5: Evidence Summary ===")

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = sum(1 for r in results if r["status"] == "FAIL")
score = round(100 * passed / total, 1) if total > 0 else 0.0

print(f"\n{'='*60}")
print(f"  NOTEBOOK: {NOTEBOOK_PATH}")
print(f"  PROJECT:  {PROJECT}")
print(f"  FILE:     web/server.py")
print(f"  TOTAL:    {total} probes")
print(f"  PASSED:   {passed}")
print(f"  FAILED:   {failed}")
print(f"  SCORE:    {score}%  (min required: {MIN_SCORE}%)")
print(f"{'='*60}")

if failed > 0:
    print("\nFAILURES:")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['id']} — {r['name']}")
            if r["detail"]:
                print(f"        {r['detail']}")

# Seal
evidence = {
    "notebook": NOTEBOOK_PATH,
    "northstar": NORTHSTAR,
    "project": PROJECT,
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "total": total,
    "passed": passed,
    "failed": failed,
    "score_pct": score,
    "min_score": MIN_SCORE,
    "verdict": "PASS" if score >= MIN_SCORE else "FAIL",
    "results": results,
}

seal = hashlib.sha256(json.dumps(evidence, sort_keys=True).encode()).hexdigest()[:16]
evidence["seal"] = seal

print(f"\n  VERDICT: {evidence['verdict']}")
print(f"  SEAL:    {seal}")
print(f"  AUTH:    65537")

=== P5: Evidence Summary ===

  NOTEBOOK: notebooks/qa/31-webservice-endpoints-qa.ipynb
  PROJECT:  solace-browser
  FILE:     web/server.py
  TOTAL:    24 probes
  PASSED:   24
  FAILED:   0
  SCORE:    100.0%  (min required: 70%)

  VERDICT: PASS
  SEAL:    78cb936864a3b866
  AUTH:    65537
